# SAM2.1 Field Boundary Prediction -- SGA Test Images

Uses local weights weights/sam2.1_hiera_large.pt.

| Sensor | Strategy |
|---|---|
| Drone (~1600x1480 px) | Resize longest side to 1024 px, single pass |
| Satellite (~3250x3266 px) | 1024x1024 patches with 128 px overlap, merge |

**Pipeline:** SAM2 inference -> morphological opening -> watershed -> polygonize -> overlay

In [ ]:
import sys, os

# SAM2 repo at ftw-baselines/sam2/ -- insert repo root so 'import sam2'
# resolves to sam2/sam2/ (the package) not the repo folder.
_SAM2_REPO = os.path.normpath(os.path.join(os.path.abspath('..'), 'sam2'))
if _SAM2_REPO not in sys.path:
    sys.path.insert(0, _SAM2_REPO)

# plot_utils.py lives in the notebooks directory
_NB_DIR = os.path.abspath('.')
if _NB_DIR not in sys.path:
    sys.path.insert(0, _NB_DIR)

import warnings, contextlib
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.colors import ListedColormap
from PIL import Image
import pandas as pd
from scipy.ndimage import binary_erosion

import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
from rasterio.features import shapes as rio_shapes
import geopandas as gpd
from shapely.geometry import shape as shapely_shape

import torch
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from plot_utils import morphological_opening, watershed_segmentation

print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device : {DEVICE}')

In [ ]:
REPO_ROOT     = os.path.abspath('..')
TEST_DATA_DIR = os.path.join(REPO_ROOT, 'data', 'sga-test')
DRONE_DIR     = os.path.join(TEST_DATA_DIR, 'drone')
SATELLITE_DIR = os.path.join(TEST_DATA_DIR, 'satellite')
OUTPUT_DIR    = os.path.join(REPO_ROOT, 'outputs', 'sam2_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAM2_CKPT   = os.path.join(REPO_ROOT, 'weights', 'sam2.1_hiera_large.pt')
SAM2_CONFIG = 'configs/sam2.1/sam2.1_hiera_l.yaml'

SGA_GROUPS = {
    'drone':     {'dir': DRONE_DIR,     'files': ['C081_orginal.jpg', 'H6_orginal.jpg', 'S10_orginal.jpg']},
    'satellite': {'dir': SATELLITE_DIR, 'files': ['demo_area01.jpg', 'demo_area02.jpg']},
}

# Inference settings
DRONE_MAX_SIDE      = 1024
SAT_PATCH_SIZE      = 1024
SAT_PATCH_OVERLAP   = 128
DRONE_GRID          = 12
SAT_PATCH_GRID      = 8
IOU_DEDUP_THRESHOLD = 0.7
MIN_MASK_AREA_RATIO = 0.005
BOUNDARY_DILATION   = 2
MIN_SCORE_THRESHOLD = 0.9

# Post-processing settings
MORPH_KERNEL     = 2    # disk radius for morphological opening
WATERSHED_KERNEL = 5    # min_distance for watershed peak detection
POLY_MIN_SIZE    = 500  # m2 -- minimum polygon area to keep

# Dummy geospatial parameters (EPSG:32632 UTM, 10 m pixels)
# Required by rasterio-based post-processing; coordinates are not real-world.
PIXEL_SIZE_M = 10.0
ORIGIN_X     = 500_000.0
ORIGIN_Y     = 5_400_000.0
TARGET_CRS   = CRS.from_epsg(32632)

CMAP = ListedColormap(['#888888', '#4CAF50', '#F44336'])
LEGEND_PATCHES = [
    mpatches.Patch(color='#888888', label='Background (0)'),
    mpatches.Patch(color='#4CAF50', label='Field interior (1)'),
    mpatches.Patch(color='#F44336', label='Boundary (2)'),
]

print(f'Checkpoint : {SAM2_CKPT}')
print(f'Output dir : {OUTPUT_DIR}')

## 1. Load SGA Test Images

In [ ]:
images = {}

for group_name, info in SGA_GROUPS.items():
    for fname in info['files']:
        stem = os.path.splitext(fname)[0]
        path = os.path.join(info['dir'], fname)
        rgb  = np.array(Image.open(path).convert('RGB'))
        images[stem] = {'rgb': rgb, 'group': group_name, 'path': path}
        h, w = rgb.shape[:2]
        print(f'  [{group_name:9s}] {fname}  {h}x{w} px')

print(f'Loaded {len(images)} images.')

## 2. Load SAM2.1 (local weights)

Loads weights/sam2.1_hiera_large.pt via uild_sam2 -- no internet needed.

In [ ]:
sam_model = build_sam2(SAM2_CONFIG, ckpt_path=SAM2_CKPT, device=DEVICE)
predictor = SAM2ImagePredictor(sam_model)
print(f'SAM2.1 hiera-large loaded from local weights  (device: {DEVICE})')

## 3. Helper Functions

In [ ]:
def resize_keep_aspect(rgb, max_side):
    h, w = rgb.shape[:2]
    scale = min(max_side / max(h, w), 1.0)
    if scale == 1.0:
        return rgb
    nh, nw = int(h * scale), int(w * scale)
    return np.array(Image.fromarray(rgb).resize((nw, nh), Image.BILINEAR))


def make_grid_points(h, w, n):
    xs = np.linspace(w * 0.05, w * 0.95, n, dtype=np.float32)
    ys = np.linspace(h * 0.05, h * 0.95, n, dtype=np.float32)
    xv, yv = np.meshgrid(xs, ys)
    return np.stack([xv.ravel(), yv.ravel()], axis=1)


def mask_iou(m1, m2):
    inter = (m1 & m2).sum()
    union = (m1 | m2).sum()
    return float(inter) / float(union + 1e-6)


def nms_masks(masks_scores, iou_thresh):
    order = sorted(range(len(masks_scores)), key=lambda i: masks_scores[i][1], reverse=True)
    kept  = []
    for idx in order:
        m_i = masks_scores[idx][0]
        if not any(mask_iou(m_i, masks_scores[j][0]) > iou_thresh for j in kept):
            kept.append(idx)
    return kept


def masks_to_semantic(instance_masks, h, w, boundary_px=2):
    semantic = np.zeros((h, w), dtype=np.uint8)
    struct   = np.ones((boundary_px * 2 + 1,) * 2, dtype=bool)
    for mask in instance_masks:
        m        = mask.astype(bool)
        eroded   = binary_erosion(m, structure=struct)
        boundary = m & ~eroded
        semantic[m]        = 1
        semantic[boundary] = 2
    return semantic


def predict_single(pred, rgb, grid_n, min_area_ratio, iou_thresh):
    h, w     = rgb.shape[:2]
    min_area = max(1, int(min_area_ratio * h * w))
    pred.set_image(rgb)
    grid_pts = make_grid_points(h, w, grid_n)
    raw_ms   = []
    for px, py in grid_pts:
        masks, scores, _ = pred.predict(
            point_coords=np.array([[px, py]], dtype=np.float32),
            point_labels=np.array([1], dtype=np.int32),
            multimask_output=True,
        )
        best  = int(scores.argmax())
        mask  = masks[best].astype(bool)
        score = float(scores[best])
        if mask.sum() >= min_area and score >= MIN_SCORE_THRESHOLD:
            raw_ms.append((mask, score))
    kept_idx = nms_masks(raw_ms, iou_thresh)
    return [raw_ms[i][0] for i in kept_idx]


def predict_satellite_patches(pred, rgb, patch_size, overlap, grid_n,
                               min_area_ratio, iou_thresh, boundary_px):
    h, w    = rgb.shape[:2]
    stride  = patch_size - overlap
    semantic = np.zeros((h, w), dtype=np.uint8)

    def tile_starts(size, ps, st):
        starts = list(range(0, size - ps, st))
        starts.append(max(0, size - ps))
        return sorted(set(starts))

    ys    = tile_starts(h, patch_size, stride)
    xs    = tile_starts(w, patch_size, stride)
    total = len(ys) * len(xs)
    done  = 0

    for y0 in ys:
        y1    = min(y0 + patch_size, h)
        for x0 in xs:
            x1    = min(x0 + patch_size, w)
            patch = rgb[y0:y1, x0:x1]
            ph, pw = patch.shape[:2]

            kept      = predict_single(pred, patch, grid_n, min_area_ratio, iou_thresh)
            patch_sem = masks_to_semantic(kept, ph, pw, boundary_px)

            vy0 = overlap // 2 if y0 > 0 else 0
            vy1 = ph - overlap // 2 if y1 < h else ph
            vx0 = overlap // 2 if x0 > 0 else 0
            vx1 = pw - overlap // 2 if x1 < w else pw
            semantic[y0 + vy0 : y0 + vy1,
                     x0 + vx0 : x0 + vx1] = patch_sem[vy0:vy1, vx0:vx1]

            done += 1
            print(f'  patch {done}/{total}  [{y0}:{y1}, {x0}:{x1}]  fields={len(kept)}')

    return semantic


def save_semantic_geotiff(semantic, out_tif):
    h, w      = semantic.shape
    transform = from_origin(ORIGIN_X, ORIGIN_Y, PIXEL_SIZE_M, PIXEL_SIZE_M)
    profile   = {
        'driver': 'GTiff', 'dtype': 'uint8', 'width': w, 'height': h,
        'count': 1, 'crs': TARGET_CRS, 'transform': transform, 'compress': 'lzw',
    }
    with rasterio.open(out_tif, 'w', **profile) as dst:
        dst.write(semantic[np.newaxis, :])


def semantic_to_rgb(semantic):
    out = np.zeros((*semantic.shape, 3), dtype=np.uint8)
    out[semantic == 0] = [136, 136, 136]
    out[semantic == 1] = [ 76, 175,  80]
    out[semantic == 2] = [244,  67,  54]
    return out


def polygonize_morph_tif(morph_tif, min_size_m2):
    with rasterio.open(morph_tif) as src:
        data      = src.read(1)
        transform = src.transform
        crs       = src.crs
    field_mask = (data == 1).astype('uint8')
    geoms = [
        shapely_shape(geom)
        for geom, val in rio_shapes(field_mask, mask=field_mask, transform=transform)
        if val == 1
    ]
    if not geoms:
        return gpd.GeoDataFrame(geometry=[], crs=crs)
    gdf = gpd.GeoDataFrame(geometry=geoms, crs=crs)
    gdf = gdf[gdf.geometry.area >= min_size_m2].reset_index(drop=True)
    return gdf


def geom_to_pixel_patches(geom, origin_x, origin_y, pixel_size):
    polys = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
    patches = []
    for poly in polys:
        xs, ys = poly.exterior.xy
        px = [(x - origin_x) / pixel_size for x in xs]
        py = [(origin_y - y) / pixel_size  for y in ys]
        patches.append(MplPolygon(list(zip(px, py))))
    return patches


print('Helpers ready.')

## 4. Inference

| Sensor | Pre-process | SAM2 strategy |
|---|---|---|
| Drone | Resize longest side to 1024 px | Single pass, 12x12 grid |
| Satellite | Native resolution | 1024x1024 patches, 8x8 grid each |


In [ ]:
results = {}  # stem -> {semantic, display_rgb, pred_tif}

if DEVICE == 'cuda':
    autocast_ctx = lambda: torch.autocast('cuda', dtype=torch.bfloat16)
else:
    autocast_ctx = contextlib.nullcontext

with torch.inference_mode(), autocast_ctx():
    for stem, info in images.items():
        rgb_orig = info['rgb']
        group    = info['group']
        h0, w0   = rgb_orig.shape[:2]

        if group == 'drone':
            rgb_inf = resize_keep_aspect(rgb_orig, DRONE_MAX_SIDE)
            h, w    = rgb_inf.shape[:2]
            print(f'\n[drone    ] {stem}  {h0}x{w0} -> {h}x{w}  grid={DRONE_GRID}x{DRONE_GRID}')
            kept     = predict_single(predictor, rgb_inf, DRONE_GRID,
                                      MIN_MASK_AREA_RATIO, IOU_DEDUP_THRESHOLD)
            semantic = masks_to_semantic(kept, h, w, BOUNDARY_DILATION)
            display_rgb = rgb_inf
            print(f'  fields detected: {len(kept)}')
        else:
            rgb_inf = rgb_orig
            h, w    = rgb_inf.shape[:2]
            print(f'\n[satellite] {stem}  {h}x{w}  '
                  f'patch={SAT_PATCH_SIZE} overlap={SAT_PATCH_OVERLAP}  '
                  f'grid={SAT_PATCH_GRID}x{SAT_PATCH_GRID}')
            semantic    = predict_satellite_patches(
                predictor, rgb_inf,
                SAT_PATCH_SIZE, SAT_PATCH_OVERLAP,
                SAT_PATCH_GRID, MIN_MASK_AREA_RATIO,
                IOU_DEDUP_THRESHOLD, BOUNDARY_DILATION,
            )
            display_rgb = rgb_inf

        # Save raw semantic GeoTIFF (needed by morph/watershed/polygonize)
        pred_tif = os.path.join(OUTPUT_DIR, f'{stem}_pred.tif')
        save_semantic_geotiff(semantic, pred_tif)

        # Save coloured RGB PNG for inspection
        Image.fromarray(semantic_to_rgb(semantic)).save(
            os.path.join(OUTPUT_DIR, f'{stem}_sam2_semantic.png')
        )

        results[stem] = {'semantic': semantic, 'display_rgb': display_rgb, 'pred_tif': pred_tif}
        print(f'  Saved: {os.path.basename(pred_tif)}')

print('\nAll images processed.')

## 5. Raw SAM2 Predictions

Original RGB | semantic mask | colour overlay

In [ ]:
n = len(results)
fig, axes = plt.subplots(n, 3, figsize=(18, 6 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for row, (stem, res) in enumerate(results.items()):
    rgb      = res['display_rgb']
    semantic = res['semantic']
    group    = images[stem]['group']

    overlay = np.zeros((*semantic.shape, 4), dtype=np.float32)
    overlay[semantic == 1] = [0.3, 0.8, 0.3, 0.45]
    overlay[semantic == 2] = [0.9, 0.2, 0.2, 0.70]

    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f'{stem} [{group}]\nOriginal RGB', fontsize=9)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(semantic, cmap=CMAP, vmin=0, vmax=2, interpolation='nearest')
    axes[row, 1].set_title(f'{stem}\nSAM2 Semantic', fontsize=9)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(rgb)
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f'{stem}\nSAM2 Overlay', fontsize=9)
    axes[row, 2].axis('off')

fig.legend(handles=LEGEND_PATCHES, loc='lower center', ncol=3, fontsize=11)
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig(os.path.join(OUTPUT_DIR, 'sam2_raw_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Post-Processing

Two stages chained sequentially (same as 	est_results.ipynb):

1. **Morphological opening** (kernel_size=2) -- erode then dilate the field class to
   remove small noise blobs.
2. **Watershed segmentation** (kernel_size=5) -- distance transform + local maxima to
   separate touching field instances into individually-labelled regions (uint16).

Both functions from plot_utils.py operate on GeoTIFF files saved in the previous step.

In [ ]:
morph_tif_paths = {}
wshed_tif_paths = {}

n = len(results)
fig, axes = plt.subplots(n, 4, figsize=(22, 6 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for row, (stem, res) in enumerate(results.items()):
    pred_tif  = res['pred_tif']
    morph_tif = os.path.join(OUTPUT_DIR, f'{stem}_morph.tif')
    wshed_tif = os.path.join(OUTPUT_DIR, f'{stem}_watershed.tif')

    morph = morphological_opening(pred_tif,  morph_tif, kernel_size=MORPH_KERNEL)
    wshed = watershed_segmentation(morph_tif, wshed_tif, kernel_size=WATERSHED_KERNEL)

    morph_tif_paths[stem] = morph_tif
    wshed_tif_paths[stem] = wshed_tif

    rgb      = res['display_rgb']
    semantic = res['semantic']

    panels = [
        (rgb,      f'{stem}\nOriginal RGB',                  {}),
        (semantic, 'SAM2 Raw',                               {'cmap': CMAP, 'vmin': 0, 'vmax': 2, 'interpolation': 'nearest'}),
        (morph,    f'Morph Opening (k={MORPH_KERNEL})',      {'cmap': CMAP, 'vmin': 0, 'vmax': 2, 'interpolation': 'nearest'}),
        (wshed,    f'Watershed (k={WATERSHED_KERNEL})',       {'interpolation': 'nearest'}),
    ]
    for col, (data, title, kw) in enumerate(panels):
        axes[row, col].imshow(data, **kw)
        axes[row, col].set_title(title, fontsize=9)
        axes[row, col].axis('off')

    print(f'{stem}: morph field px={int((morph==1).sum())}  watershed instances={int(wshed.max())}')

fig.legend(handles=LEGEND_PATCHES, loc='lower center', ncol=3, fontsize=11)
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig(os.path.join(OUTPUT_DIR, 'postprocessing_results.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Polygonisation

Converts the morphological prediction TIF (class labels 0/1/2) to vector polygons
using rasterio.features.shapes directly.

**Parameters:**
- POLY_MIN_SIZE=500 m2 -- drop polygons smaller than 500 m2 (5 pixels at 10 m)
- Dummy EPSG:32632 CRS; polygon coordinates and areas have no real-world meaning.

In [ ]:
polygon_results = {}  # stem -> GeoDataFrame

for stem, morph_tif in morph_tif_paths.items():
    gdf = polygonize_morph_tif(morph_tif, min_size_m2=POLY_MIN_SIZE)
    polygon_results[stem] = gdf
    print(f'  {stem}: {len(gdf)} polygons detected')

print('Polygonisation complete.')

## 8. Polygon Overlay on Original Image

Field polygons (from morphological prediction) drawn over the original RGB.
Green fill = field interior, red edge = boundary.

In [ ]:
n = len(polygon_results)
fig, axes = plt.subplots(n, 2, figsize=(16, 7 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for row, (stem, gdf) in enumerate(polygon_results.items()):
    rgb  = results[stem]['display_rgb']
    h, w = rgb.shape[:2]

    # Col 0: original RGB
    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f'{stem} [{images[stem]["group"]}]\nOriginal RGB', fontsize=9)
    axes[row, 0].axis('off')

    # Col 1: polygon overlay
    axes[row, 1].imshow(rgb)
    if len(gdf) > 0:
        patches = []
        for geom in gdf.geometry:
            if geom is None or geom.is_empty:
                continue
            patches.extend(geom_to_pixel_patches(geom, ORIGIN_X, ORIGIN_Y, PIXEL_SIZE_M))
        if patches:
            pc = PatchCollection(
                patches,
                facecolor='#4CAF5033',
                edgecolor='#F44336',
                linewidth=0.8,
                match_original=False,
            )
            axes[row, 1].add_collection(pc)
    axes[row, 1].set_xlim(0, w)
    axes[row, 1].set_ylim(h, 0)  # origin top-left
    axes[row, 1].set_title(f'{stem}\nField Polygons (n={len(gdf)})', fontsize=9)
    axes[row, 1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'polygon_overlay.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Per-Image Statistics

In [ ]:
rows = []
for stem, res in results.items():
    sem   = res['semantic']
    total = sem.size
    h, w  = sem.shape
    gdf   = polygon_results.get(stem)
    rows.append({
        'Image':            stem,
        'Group':            images[stem]['group'],
        'H x W (inferred)': f'{h}x{w}',
        'Field px %':       f'{100*(sem==1).sum()/total:.1f}',
        'Boundary px %':    f'{100*(sem==2).sum()/total:.1f}',
        'Polygons':         len(gdf) if gdf is not None else 0,
    })
display(pd.DataFrame(rows))

## 10. Sensor-Type Comparison

In [ ]:
for group_label, color in [('drone', '#FF6F00'), ('satellite', '#1565C0')]:
    group_stems = [s for s in results if images[s]['group'] == group_label]
    n_cols = len(group_stems)
    fig, axes = plt.subplots(2, n_cols, figsize=(5.5 * n_cols, 11))
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    for col, stem in enumerate(group_stems):
        rgb      = results[stem]['display_rgb']
        h, w     = rgb.shape[:2]
        gdf      = polygon_results.get(stem)

        axes[0, col].imshow(rgb)
        axes[0, col].set_title(f'{stem}\n{h}x{w} px', fontsize=8)
        axes[0, col].axis('off')

        axes[1, col].imshow(rgb)
        if gdf is not None and len(gdf) > 0:
            patches = []
            for geom in gdf.geometry:
                if geom is not None and not geom.is_empty:
                    patches.extend(geom_to_pixel_patches(geom, ORIGIN_X, ORIGIN_Y, PIXEL_SIZE_M))
            if patches:
                pc = PatchCollection(patches, facecolor='#4CAF5033',
                                     edgecolor='#F44336', linewidth=0.8, match_original=False)
                axes[1, col].add_collection(pc)
        axes[1, col].set_xlim(0, w)
        axes[1, col].set_ylim(h, 0)
        axes[1, col].set_title(f'{stem}\n{len(gdf) if gdf is not None else 0} polygons', fontsize=8)
        axes[1, col].axis('off')

    axes[0, 0].set_ylabel('Original RGB',     fontsize=9, color=color, labelpad=6)
    axes[1, 0].set_ylabel('Polygon Overlay',  fontsize=9, color=color, labelpad=6)
    plt.suptitle(f'SAM2.1 -- {group_label.title()} Images', fontsize=12, color=color)
    plt.tight_layout(rect=[0, 0.0, 1, 1])
    plt.savefig(os.path.join(OUTPUT_DIR, f'sam2_{group_label}_comparison.png'),
                dpi=150, bbox_inches='tight')
    plt.show()